# Tiền xử lý FAIDSet  ELT Pipeline

```
E  Extract:   đọc .txt từ raw_data/
L  Load:      gộp thành DataFrame, lưu processed_data/full.csv
T  Transform: làm sạch, filter, split  train/val/test.csv
```

In [1]:
!pip install pandas scikit-learn -q


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## E  Extract
Đọc toàn bộ file `.txt` từ `raw_data/`, gắn nhãn `language` và `label`.

In [2]:
from pathlib import Path
import pandas as pd

RAW = Path("raw_data")

# Mapping folder  (language, label)
FOLDER_MAP = {
    ("eng", "AI"):    {"language": "en", "label": 1},
    ("eng", "human"): {"language": "en", "label": 0},
    ("vi",  "AI"):    {"language": "vi", "label": 1},
    ("vi",  "human"): {"language": "vi", "label": 0},
}

records = []
for (lang, kind), meta in FOLDER_MAP.items():
    folder = RAW / lang / kind
    files  = list(folder.glob("*.txt"))
    for f in files:
        records.append({
            "file":     str(f),
            "text":     f.read_text(encoding="utf-8"),
            "language": meta["language"],
            "label":    meta["label"],   # 1=AI, 0=human
        })
    print(f"raw_data/{lang}/{kind}/    {len(files):,} files")

df_raw = pd.DataFrame(records)
print(f"\nTổng: {len(df_raw):,} records")
print(df_raw[["language", "label"]].value_counts().sort_index())

raw_data/eng/AI/    10,423 files
raw_data/eng/human/    7,166 files
raw_data/vi/AI/    9,161 files
raw_data/vi/human/    13,069 files

Tổng: 39,819 records
language  label
en        0         7166
          1        10423
vi        0        13069
          1         9161
Name: count, dtype: int64


---
## L  Load
Lưu toàn bộ data thô (chưa transform) vào `processed_data/full.csv`.

In [3]:
PROCESSED = Path("processed_data")
PROCESSED.mkdir(exist_ok=True)

df_raw.to_csv(PROCESSED / "full.csv", index=False, encoding="utf-8-sig")

print(f" Đã lưu: processed_data/full.csv  ({len(df_raw):,} rows)")
print(f"\nPreview:")
print(df_raw[["language", "label", "text"]].head(3).to_string())

 Đã lưu: processed_data/full.csv  (39,819 rows)

Preview:
  language  label                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

---
## T  Transform
Làm sạch  filter  split  lưu `train/val/test.csv`.

In [4]:
#  Bước 1: Load lại từ full.csv 
df = pd.read_csv(PROCESSED / "full.csv", encoding="utf-8-sig")
print(f"Đọc vào: {len(df):,} records")

Đọc vào: 39,819 records


In [5]:
import re

#  Bước 2: Làm sạch text 
def clean_text(text):
    text = str(text)
    text = re.sub(r'\s+', ' ', text)          # nhiều khoảng trắng  1
    text = re.sub(r'[^\S\n]+', ' ', text)     # tab, non-breaking space
    text = text.strip()
    return text

df["text"] = df["text"].apply(clean_text)
print("Đã làm sạch text")

Đã làm sạch text


In [6]:
#  Bước 3: Filter 
# Xóa text quá ngắn (< 50 ký tự) hoặc rỗng
MIN_CHARS = 50

before = len(df)
df = df[df["text"].str.len() >= MIN_CHARS].reset_index(drop=True)
after  = len(df)

print(f"Filter text < {MIN_CHARS} ký tự: {before - after} rows bị xóa")
print(f"   Còn lại: {after:,} records")
print(f"\nPhân phối sau filter:")
print(df.groupby(["language", "label"]).size().to_string())

Filter text < 50 ký tự: 2 rows bị xóa
   Còn lại: 39,817 records

Phân phối sau filter:
language  label
en        0         7166
          1        10423
vi        0        13067
          1         9161


In [7]:
from sklearn.model_selection import train_test_split

#  Bước 4: Train / Val / Test split  (70 / 15 / 15) 
# Stratify theo cả language + label để cân bằng
df["stratify_key"] = df["language"] + "_" + df["label"].astype(str)

df_train, df_temp = train_test_split(
    df, test_size=0.30, random_state=42, stratify=df["stratify_key"]
)
df_val, df_test = train_test_split(
    df_temp, test_size=0.50, random_state=42, stratify=df_temp["stratify_key"]
)

# Xóa cột tạm
for d in [df_train, df_val, df_test]:
    d.drop(columns=["stratify_key"], inplace=True)

print(f"Train: {len(df_train):,}  |  Val: {len(df_val):,}  |  Test: {len(df_test):,}")

Train: 27,871  |  Val: 5,973  |  Test: 5,973


In [8]:
#  Bước 5: Lưu kết quả 
df_train.to_csv(PROCESSED / "train.csv", index=False, encoding="utf-8-sig")
df_val.to_csv  (PROCESSED / "val.csv",   index=False, encoding="utf-8-sig")
df_test.to_csv (PROCESSED / "test.csv",  index=False, encoding="utf-8-sig")

print("Đã lưu:")
print(f"   processed_data/train.csv  ({len(df_train):,} rows)")
print(f"   processed_data/val.csv    ({len(df_val):,} rows)")
print(f"   processed_data/test.csv   ({len(df_test):,} rows)")

print("\nPhân phối train:")
print(df_train.groupby(["language", "label"]).size().to_string())
print("\nPhân phối val:")
print(df_val.groupby(["language", "label"]).size().to_string())
print("\nPhân phối test:")
print(df_test.groupby(["language", "label"]).size().to_string())

Đã lưu:
   processed_data/train.csv  (27,871 rows)
   processed_data/val.csv    (5,973 rows)
   processed_data/test.csv   (5,973 rows)

Phân phối train:
language  label
en        0        5016
          1        7296
vi        0        9147
          1        6412

Phân phối val:
language  label
en        0        1075
          1        1563
vi        0        1960
          1        1375

Phân phối test:
language  label
en        0        1075
          1        1564
vi        0        1960
          1        1374
